In [23]:
import re
import pandas as pd
import unicodedata

In [19]:
df = pd.read_csv('llm7.csv', encoding_errors='ignore')
df.head(2)

,text,label
0,Cars. Cars have been around since they became ...,0
1,Transportation is a large necessity in most co...,0


In [15]:
words = ' '.join(df.text.to_list()).lower().split()
words = list(dict.fromkeys(words))
chars = sorted(list(set(''.join(df.text.to_list()).lower())))

In [18]:
chars

['\n',
 ' ',
 '!',
 '"',
 '#',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '<',
 '=',
 '>',
 '?',
 '[',
 '\\',
 ']',
 '^',
 '_',
 '`',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z',
 '{',
 '|',
 '}',
 '~',
 '\x80',
 '\x85',
 '\x91',
 '\x92',
 '\x93',
 '\x94',
 '\x97',
 '\x99',
 '\xa0',
 '¢',
 '£',
 '¨',
 '©',
 '®',
 '°',
 '²',
 '´',
 '¶',
 '¹',
 'á',
 'ä',
 'å',
 'é',
 'ë',
 'ó',
 'ö',
 'þ',
 'ā',
 '–',
 '—',
 '‘',
 '’',
 '“',
 '”',
 '─',
 '😁',
 '😊',
 '😜']

In [21]:
dfx = df.copy()
text = ' '.join(df.text.to_list())

In [22]:
# 1. Repair mojibake (turns \x91-\x99, \x80, \x85 back into real chars)
CP1252_FIXES = {
    "\x80": "€",  "\x85": "…",  "\x91": "'",  "\x92": "'",
    "\x93": '"',  "\x94": '"',  "\x97": "—",  "\x99": "™",
}
for bad, good in CP1252_FIXES.items():
    text = text.replace(bad, good)

In [24]:
# 2. Unicode-normalise: converts ² -> 2, full-width chars -> ASCII, etc.
text = unicodedata.normalize("NFKC", text)

In [25]:
# 3. Map typographic chars to ASCII equivalents
table = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"',
    "\u2013": "-", "\u2014": "-", "\u2012": "-", "\u2015": "-",
    "\u2026": "...", "\u00b4": "'", "\u00a8": '"',
    "\u00a0": " ",  # non-breaking space
    "\u2500": "-",  # box-drawing
})
text = text.translate(table)

In [26]:
# 4. Drop emoji, symbols, currency, marks (optional)
#    Categories: So = symbol-other, Sk = modifier, Sc = currency
text = "".join(c for c in text if unicodedata.category(c) not in {"So", "Sk", "Sc"})

In [27]:
# 5. Drop remaining control characters (Cc) except \n \t
text = "".join(
    c for c in text
    if unicodedata.category(c) != "Cc" or c in "\n\t"
)

In [28]:
# 6. Strip accents: á -> a, ö -> o
text = "".join(
    c for c in unicodedata.normalize("NFKD", text)
    if not unicodedata.combining(c)
)

In [29]:
# 7. Collapse whitespace
text = re.sub(r"\s+", " ", text).strip()

In [ ]:
# 8. Optional: drop any remaining non-ASCII
# text = text.encode("ascii", "ignore").decode()

In [31]:
set(text.lower())

{' ',
 '!',
 '"',
 '#',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '<',
 '=',
 '>',
 '?',
 '[',
 '\\',
 ']',
 '_',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z',
 '{',
 '|',
 '}',
 '~',
 '¶',
 'þ'}